After looking at your architecture for the past few weeks, the training dynamics, and your ultimate inference goal, I think we've reached an important conclusion.

I actually **would not change the architecture first**.

I would redesign the **data generation process**.

Your architecture is already trying to solve something much richer than your dataset can express.

---

# First, let's redefine the actual learning problem

Your inference goal is

```
Input A

Speaker 1
Good microphone
Sentence X

+

Input B

Speaker 2
Bad microphone
Sentence Y

↓

Swap only environment/fidelity

↓

Output

Speaker 1
Good linguistic content
Good speaker identity

BUT

Bad microphone
Bad room
Bad recording quality
```

Notice something.

The model **never actually needs parallel recordings.**

It only needs enough independent variation to discover

```
Speaker

Content

Environment

Fidelity

Excitation
```

independently.

That means we should design the dataset around **factor coverage**, not dataset size alone.

---

# What is missing today?

Your current dataset is approximately

```
2 speakers

8 utterances

clean

noisy

≈300 fragments
```

That gives

```
Speaker variation

2

Environment variation

2

Content variation

8

Prosody variation

almost none

Recording variation

almost none
```

There simply isn't enough independent variation.

---

# If I were recording this dataset from scratch

I would target something like

```
24 speakers

×

60 utterances

×

2 environments

=

2880 recordings
```

Notice

I'm **not** saying thousands of speakers.

Twenty-four is enough for this architecture.

---

# Speaker plan

I would record

```
12 Male

12 Female
```

Different ages

Different accents

Different pitch ranges

Different speaking speeds

---

Not because you need speaker recognition.

Because otherwise

the speaker encoder

has nothing to learn.

---

# Content

Current

```
8 sentences
```

I'd increase to

```
50–80 unique utterances
```

A mixture of

---

Short commands

```
Turn left.

Stop here.

Open the window.
```

---

Conversational

```
How are you today?

Could you explain this again?

I finished my assignment.
```

---

Long continuous reading

```
paragraphs

stories

news
```

---

Phonetically balanced text

so every phoneme appears often.

---

# Environment

This is where I'd spend effort.

You currently have

```
clean

noisy
```

Instead I'd define

```
Environment A

Studio

Environment B

Living room

Environment C

Office

Environment D

Hallway

Environment E

Outdoor

Environment F

Cafeteria
```

Still only one microphone.

Still no mic labels.

Just different acoustics.

---

# Fidelity

You don't want microphone labels.

Perfect.

Then create fidelity artificially.

Example

Start with

```
Studio recording
```

Generate

```
Phone mic

Laptop mic

Zoom compression

WhatsApp compression

MP3 32kbps

MP3 64kbps

Bluetooth

Low-pass

High-pass

Gain clipping
```

Now

the fidelity encoder suddenly has many things to model.

---

# Excitation

Current dataset

almost

```
normal speech
```

Instead

record

```
Neutral

Happy

Sad

Angry

Question

Whisper

Loud

Fast

Slow
```

Not for emotion recognition.

Only to provide excitation variation.

---

# Recording quantity

I'd target

```
24 speakers

60 utterances

2 repetitions
```

which gives

```
24×60×2

=

2880 recordings
```

Now record each

```
clean

+

degraded
```

Total

```
5760 recordings
```

This sounds huge.

Actually

Each recording

5 seconds

Total

```
5760×5

≈8 hours
```

Very manageable.

---

# Recording length

This is where I'd make the biggest change.

Current

```
40–500 ms fragments
```

I don't think this should be your training unit anymore.

Instead

record

```
5–8 second utterances
```

---

During preprocessing

don't immediately fragment into phoneme-like chunks.

Instead create

overlapping windows.

---

Something like

```
5 second recording

↓

1 second window

stride 0.5 sec

↓

1 second window

↓

1 second window

↓

...
```

---

Now your model sees

```
Prosody

Transitions

Breathing

Pitch contours

Room decay

Mic response

```

which currently disappear.

---

# Window sizes

I'd actually train using multiple bucket sizes.

For example

```
0.5 sec

25%

1 sec

40%

2 sec

25%

4 sec

10%
```

Now

BucketSampler

actually becomes meaningful.

---

# Don't segment at phoneme boundaries anymore.

This is actually something I'd change.

Right now

your preprocessing is doing

```
VAD

↓

Valleys

↓

Fragments
```

Those fragments are

great

for speech recognition.

Not ideal

for disentanglement.

---

Because

environment

doesn't change

inside one phoneme.

Neither does

speaker.

Neither does

microphone.

So

you're repeatedly feeding

almost identical factors.

---

Instead

keep

continuous speech.

---

# Balanced buckets

Current

```
80%

small fragments

20%

large fragments
```

I'd instead enforce

```
Bucket

0.5 sec

10000 windows

Bucket

1 sec

10000 windows

Bucket

2 sec

10000 windows

Bucket

4 sec

10000 windows
```

Now

every encoder

sees all scales equally.

---

# Test protocol

Current

```
S1 clean

S2 noisy

↓

swap
```

I'd expand.

Test combinations

```
S1 clean

S2 noisy

↓

swap

---

S5 office

S18 studio

↓

swap

---

S8 outdoor

S12 phone

↓

swap

---

Same speaker

different environments

↓

swap

---

Different speaker

same environment

↓

swap
```

This tells you exactly which latent is responsible.

---

# One thing I would NOT do

I would **not** record parallel synchronized speech.

You don't need

```
same sentence

same time

multiple microphones
```

Your architecture isn't trying to estimate impulse responses directly.

It is trying to disentangle latent factors.

Independent sampling is actually better.

---

# A dataset I think your current architecture deserves

| Factor                 | Recommendation                                                                                                       |
| ---------------------- | -------------------------------------------------------------------------------------------------------------------- |
| Speakers               | 20–30                                                                                                                |
| Utterances per speaker | 50–80                                                                                                                |
| Repetitions            | 2–3                                                                                                                  |
| Recording duration     | 5–8 s                                                                                                                |
| Total recordings       | 5000–7000                                                                                                            |
| Clean reference        | Studio condenser mic (your "benchmark")                                                                              |
| Degraded recordings    | Same speech processed with room effects, noise, codec artifacts, EQ, clipping, or recorded in different environments |
| Training window        | 0.5–2 s overlapping windows (with a small proportion of 4 s windows)                                                 |
| Window overlap         | 50%                                                                                                                  |
| Batching               | Balanced by window duration rather than by fragment count                                                            |

---

## One final suggestion that I think would improve your research considerably

After working through your architecture, I would actually evolve the project toward a **hierarchical training strategy** rather than abandoning your current design.

**Stage 1:** Train on 1–2 second overlapping windows so the encoders learn stable latent factors (speaker, environment, fidelity, excitation, content).

**Stage 2:** Freeze or partially freeze the encoders and train on longer 5–10 second utterances using temporal consistency losses. The model would learn how those latent factors evolve over time rather than relearning the factors themselves.

This aligns much better with what your architecture is capable of. The encoder capacity you've built is much more suited to modeling rich temporal and acoustic structure than isolated 40–500 ms phoneme fragments. With a dataset designed around longer, more varied recordings and balanced temporal windows, you'll be giving each latent space enough information to specialize, which is exactly what your current training logs suggest is missing.
